In [0]:
# =====================================================
# PHASE 3: AI/ML PRICE PREDICTION MODEL
# =====================================================

print("🤖 Starting ML Model Training...")

from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import *

print("✅ Libraries imported successfully!")

🤖 Starting ML Model Training...
✅ Libraries imported successfully!


In [0]:
# Load the cleaned data from Phase 1
df = spark.table("hackathon_db.car_sales_clean")

print(f"📊 Loaded {df.count()} records for training")
print(f"📋 Columns: {df.columns}")

# Show sample data
display(df.select("brand", "model", "year", "mileage", "price", "fuel_type", "engine_size", "car_age").limit(5))

📊 Loaded 32385 records for training
📋 Columns: ['brand', 'model', 'engine_size', 'fuel_type', 'year', 'mileage', 'price', 'car_age', 'price_category', 'mileage_category']


brand,model,year,mileage,price,fuel_type,engine_size,car_age
VW,Polo,2011,72003,10885,Petrol,1.6,14
Porsche,Cayenne,2003,145023,11072,Diesel,2.6,22
Ford,Focus,2018,28420,37763,Diesel,2.0,7
BMW,Z4,2022,5046,58818,Petrol,2.0,3
Toyota,Yaris,2018,18095,28278,Hybrid,1.2,7


In [0]:
# Handle categorical variables (convert text to numbers)
print("🔧 Encoding categorical features...")

indexers = [
    StringIndexer(inputCol="brand", outputCol="brand_indexed", handleInvalid="keep"),
    StringIndexer(inputCol="model", outputCol="model_indexed", handleInvalid="keep"),
    StringIndexer(inputCol="fuel_type", outputCol="fuel_type_indexed", handleInvalid="keep")
]

# Combine all features into one vector
numeric_features = ["year", "mileage", "car_age", "engine_size"]
categorical_features = ["brand_indexed", "model_indexed", "fuel_type_indexed"]
all_features = numeric_features + categorical_features

assembler = VectorAssembler(
    inputCols=all_features,
    outputCol="features",
    handleInvalid="skip"
)

# Scale features (normalize them)
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)

print("✅ Feature engineering setup complete!")

🔧 Encoding categorical features...
✅ Feature engineering setup complete!


In [0]:
# Split data into training (80%) and testing (20%)
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

print(f"📚 Training set: {train_data.count()} cars")
print(f"🧪 Test set: {test_data.count()} cars")

# Create Random Forest model
print("\n🌲 Building Random Forest model...")

rf = RandomForestRegressor(
    featuresCol="scaled_features",
    labelCol="price",
    numTrees=100,
    maxDepth=10,
    seed=42
)

# Create ML Pipeline (combines all steps)
pipeline = Pipeline(stages=indexers + [assembler, scaler, rf])

print("✅ Model pipeline created!")

📚 Training set: 25852 cars
🧪 Test set: 6533 cars

🌲 Building Random Forest model...
✅ Model pipeline created!


In [0]:
# Train the model
print("🚀 Training model... (this takes 2-5 minutes)")
print("☕ Grab a coffee while this runs!")

model = pipeline.fit(train_data)

print("\n✅ Model training complete! 🎉")

🚀 Training model... (this takes 2-5 minutes)
☕ Grab a coffee while this runs!

✅ Model training complete! 🎉


In [0]:
# Make predictions on test data
print("🔮 Making predictions on test data...")

predictions = model.transform(test_data)

# Show sample predictions
print("\n=== Sample Predictions ===")
display(
    predictions.select(
        "brand", "model", "year", "mileage", 
        "price", 
        round(col("prediction"), 2).alias("predicted_price"),
        round((col("price") - col("prediction")), 2).alias("difference")
    ).limit(10)
)

🔮 Making predictions on test data...

=== Sample Predictions ===


brand,model,year,mileage,price,predicted_price,difference
BMW,M5,2000,134052,14725,15462.6,-737.6
BMW,M5,2000,153596,12574,13691.36,-1117.36
BMW,M5,2000,161667,11760,12493.37,-733.37
BMW,M5,2001,102097,20097,20459.01,-362.01
BMW,M5,2001,123766,17062,18363.16,-1301.16
BMW,M5,2001,127091,16626,17450.09,-824.09
BMW,M5,2001,137939,15261,16152.43,-891.43
BMW,M5,2001,140583,14941,16152.43,-1211.43
BMW,M5,2001,216712,7796,11068.24,-3272.24
BMW,M5,2002,90864,23303,23455.47,-152.47


In [0]:
# Calculate model accuracy metrics
print("📊 Evaluating Model Performance...\n")

# RMSE (Root Mean Squared Error) - lower is better
rmse_evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
rmse = rmse_evaluator.evaluate(predictions)

# MAE (Mean Absolute Error) - average prediction error
mae_evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="mae")
mae = mae_evaluator.evaluate(predictions)

# R² Score (how well model fits data) - closer to 1.0 is better
r2_evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2")
r2 = r2_evaluator.evaluate(predictions)

print("=" * 50)
print("🎯 MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"📉 RMSE (Root Mean Squared Error): ${rmse:,.2f}")
print(f"📊 MAE (Mean Absolute Error): ${mae:,.2f}")
print(f"🎯 R² Score (Accuracy): {r2:.4f} ({r2*100:.2f}%)")
print("=" * 50)

if r2 > 0.80:
    print("✅ EXCELLENT! Model is highly accurate!")
elif r2 > 0.70:
    print("✅ GOOD! Model performs well!")
else:
    print("⚠️ Model needs improvement, but shows promise!")

📊 Evaluating Model Performance...

🎯 MODEL PERFORMANCE METRICS
📉 RMSE (Root Mean Squared Error): $1,483.29
📊 MAE (Mean Absolute Error): $983.87
🎯 R² Score (Accuracy): 0.9928 (99.28%)
✅ EXCELLENT! Model is highly accurate!


In [0]:
# Show which features are most important for predictions
print("\n🔍 Feature Importance Analysis")
print("=" * 50)

rf_model = model.stages[-1]  # Get the Random Forest model
feature_importance = rf_model.featureImportances

# Create a nice display
importance_list = []
for i, importance in enumerate(feature_importance):
    if i < len(all_features):
        importance_list.append((all_features[i], float(importance)))

# Sort by importance
importance_list.sort(key=lambda x: x[1], reverse=True)

print("\nTop factors affecting car prices:")
for i, (feature, importance) in enumerate(importance_list[:8], 1):
    bar = "█" * int(importance * 50)
    print(f"{i}. {feature:20s} {bar} {importance:.4f}")

print("=" * 50)


🔍 Feature Importance Analysis

Top factors affecting car prices:
1. year                 ███████████ 0.2343
2. engine_size          ██████████ 0.2163
3. car_age              ██████████ 0.2056
4. model_indexed        ████████ 0.1625
5. mileage              █████ 0.1152
6. brand_indexed        ██ 0.0485
7. fuel_type_indexed     0.0175


In [0]:
# Create a function to predict any car's price
def predict_car_price(brand, model_name, year, mileage, fuel_type, engine_size):
    """
    Predict the price of a car based on its features
    """
    car_age = 2025 - year
    
    new_data = spark.createDataFrame([
        (brand, model_name, year, mileage, fuel_type, engine_size, car_age)
    ], ["brand", "model", "year", "mileage", "fuel_type", "engine_size", "car_age"])
    
    prediction = model.transform(new_data)
    predicted_price = prediction.select("prediction").first()[0]
    
    return predicted_price

# Test predictions on sample cars
print("\n🚗 LIVE PRICE PREDICTIONS")
print("=" * 60)

test_cars = [
    ("Toyota", "Camry", 2020, 35000, "Petrol", 2.0),
    ("Honda", "Civic", 2019, 45000, "Petrol", 1.8),
    ("BMW", "3 Series", 2022, 15000, "Diesel", 2.5),
    ("Ford", "Focus", 2018, 60000, "Petrol", 1.6),
    ("Mercedes", "C-Class", 2021, 25000, "Diesel", 2.2),
]

print("Testing predictions with sample cars...\n")

for brand, model_name, year, mileage, fuel, engine in test_cars:
    try:
        predicted = predict_car_price(brand, model_name, year, mileage, fuel, engine)
        print(f"🚙 {brand} {model_name} ({year})")
        print(f"   Mileage: {mileage:,} km | Engine: {engine}L")
        print(f"   💰 Predicted Price: ${predicted:,.2f}")
        print("-" * 60)
    except Exception as e:
        print(f"🚙 {brand} {model_name} ({year})")
        print(f"   ⚠️ Prediction unavailable")
        print("-" * 60)

print("=" * 60)

# Now analyze predictions on real data
print("\n\n🎯 PREDICTIONS ON REAL CARS FROM YOUR DATA")
print("=" * 60)

# Get sample comparisons
real_comparisons = predictions.select(
    "brand", "model", "year", "mileage", "price",
    round(col("prediction"), 2).alias("predicted_price"),
    round(abs(col("price") - col("prediction")), 2).alias("difference"),
    round((1 - abs(col("price") - col("prediction")) / col("price")) * 100, 2).alias("accuracy_pct")
).limit(5)

print("Comparing actual prices vs predicted prices:\n")
display(real_comparisons)

# Get summary statistics
print("\n📊 PREDICTION ACCURACY SUMMARY:")
print("=" * 60)

accuracy_stats = predictions.select(
    round(avg(abs(col("price") - col("prediction")) / col("price") * 100), 2).alias("avg_error_pct"),
    round(avg(abs(col("price") - col("prediction"))), 2).alias("avg_error_amount"),
    count(when(abs(col("price") - col("prediction")) / col("price") < 0.10, 1)).alias("within_10_pct"),
    count(when(abs(col("price") - col("prediction")) / col("price") < 0.20, 1)).alias("within_20_pct"),
    count("*").alias("total")
).first()

avg_error_pct = accuracy_stats.avg_error_pct
avg_error_amount = accuracy_stats.avg_error_amount
within_10 = accuracy_stats.within_10_pct
within_20 = accuracy_stats.within_20_pct
total = accuracy_stats.total

print(f"Average Error: {avg_error_pct:.1f}%")
print(f"Average $ Difference: ${avg_error_amount:,.2f}")
print(f"Predictions within 10%: {within_10}/{total} ({within_10/total*100:.1f}%)")
print(f"Predictions within 20%: {within_20}/{total} ({within_20/total*100:.1f}%)")
print("=" * 60)

if within_10/total > 0.5:
    print("✅ Over 50% of predictions are within 10% - EXCELLENT!")
elif within_20/total > 0.7:
    print("✅ Over 70% of predictions are within 20% - GOOD!")
else:
    print("⚠️ Model shows promise but needs more training data")

print("\n🎯 BEST PREDICTIONS (Most Accurate):")
print("=" * 60)

best_predictions = predictions.select(
    "brand", "model", "year", "price",
    round(col("prediction"), 2).alias("predicted_price"),
    round(abs(col("price") - col("prediction")) / col("price") * 100, 2).alias("error_pct")
).orderBy("error_pct").limit(5)

display(best_predictions)


🚗 LIVE PRICE PREDICTIONS
Testing predictions with sample cars...

🚙 Toyota Camry (2020)
   Mileage: 35,000 km | Engine: 2.0L
   💰 Predicted Price: $52,942.85
------------------------------------------------------------
🚙 Honda Civic (2019)
   Mileage: 45,000 km | Engine: 1.8L
   💰 Predicted Price: $50,702.31
------------------------------------------------------------
🚙 BMW 3 Series (2022)
   Mileage: 15,000 km | Engine: 2.5L
   💰 Predicted Price: $68,938.66
------------------------------------------------------------
🚙 Ford Focus (2018)
   Mileage: 60,000 km | Engine: 1.6L
   💰 Predicted Price: $27,214.62
------------------------------------------------------------
🚙 Mercedes C-Class (2021)
   Mileage: 25,000 km | Engine: 2.2L
   💰 Predicted Price: $73,959.21
------------------------------------------------------------


🎯 PREDICTIONS ON REAL CARS FROM YOUR DATA
Comparing actual prices vs predicted prices:



brand,model,year,mileage,price,predicted_price,difference,accuracy_pct
BMW,M5,2000,134052,14725,15462.6,737.6,94.99
BMW,M5,2000,153596,12574,13691.36,1117.36,91.11
BMW,M5,2000,161667,11760,12493.37,733.37,93.76
BMW,M5,2001,102097,20097,20459.01,362.01,98.2
BMW,M5,2001,123766,17062,18363.16,1301.16,92.37



📊 PREDICTION ACCURACY SUMMARY:
Average Error: 6.6%
Average $ Difference: $983.87
Predictions within 10%: 5252/6533 (80.4%)
Predictions within 20%: 6293/6533 (96.3%)
✅ Over 50% of predictions are within 10% - EXCELLENT!

🎯 BEST PREDICTIONS (Most Accurate):


brand,model,year,price,predicted_price,error_pct
Ford,Focus,2014,22342,22341.78,0.0
Ford,Focus,2010,12467,12467.38,0.0
Ford,Mondeo,2012,16062,16062.76,0.0
Toyota,Prius,2012,18494,18494.34,0.0
Toyota,Prius,2004,11165,11165.47,0.0


In [0]:
# Save predictions for dashboard
print("💾 Saving predictions to database...")

predictions_final = predictions.select(
    "brand",
    "model", 
    "year",
    "mileage",
    "price",
    round(col("prediction"), 2).alias("predicted_price"),
    round((col("price") - col("prediction")), 2).alias("difference"),
    round(abs(col("price") - col("prediction")) / col("price") * 100, 2).alias("error_pct")
).filter(col("price") > 1000)

predictions_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("hackathon_db.car_price_predictions")

print("✅ Predictions saved to: hackathon_db.car_price_predictions")
print(f"📊 Total predictions: {predictions_final.count()}")

# Show best predictions (lowest error)
print("\n🎯 Most Accurate Predictions:")
display(predictions_final.orderBy("error_pct").limit(10))

💾 Saving predictions to database...
✅ Predictions saved to: hackathon_db.car_price_predictions
📊 Total predictions: 6532

🎯 Most Accurate Predictions:


brand,model,year,mileage,price,predicted_price,difference,error_pct
Toyota,Prius,2012,51519,18494,18494.34,-0.34,0.0
Toyota,Prius,2004,91064,11165,11165.47,-0.47,0.0
Ford,Focus,2010,99364,12467,12467.38,-0.38,0.0
Ford,Mondeo,2012,81199,16062,16062.76,-0.76,0.0
Ford,Focus,2014,44977,22342,22341.78,0.22,0.0
VW,Passat,2018,34608,36185,36186.27,-1.27,0.0
Toyota,RAV4,2002,151260,7228,7228.71,-0.71,0.01
Toyota,Yaris,2021,13962,35318,35314.53,3.47,0.01
Toyota,Yaris,2013,61522,15629,15626.95,2.05,0.01
Ford,Mondeo,2013,63523,18091,18090.03,0.97,0.01
